# La même série, deux histoires · *The same series, two stories*

Notebook compagnon du chapitre **18. Atelier données : découvrir FRED, l'entrepôt de la Réserve fédérale** — [lire l'article](https://nmlab.io/ressources/atelier-donnees-decouvrir-fred).
Companion notebook to chapter **18. Data Workshop: Discovering FRED** — [read the article](https://nmlab.io/en/ressources/data-workshop-discovering-fred).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


from pandas import Series

def load_prices() -> tuple[Series, Series]:
    """Indice des prix (IPC) et son glissement annuel — l'inflation — depuis 1995.
    Consumer price index and its year-over-year change (inflation), since 1995."""
    cpi = nm.load_fred("CPIAUCSL").loc["1995":]
    inflation = (cpi / cpi.shift(12) - 1) * 100
    return cpi, inflation

cpi, inflation = load_prices()
print(f"Dernier point / latest: {inflation.index[-1]:%Y-%m} → {inflation.iloc[-1]:.1f} %")


import matplotlib.dates as mdates
from matplotlib.figure import Figure

MONTHS_FR = ["janvier", "février", "mars", "avril", "mai", "juin", "juillet",
             "août", "septembre", "octobre", "novembre", "décembre"]

LABELS = {
    "fr": dict(
        title="La même série, deux histoires",
        sub="Un menu déroulant transforme un niveau en taux d'inflation",
        y1="« Niveau »", y2="« Variation sur un an », %",
        cpi_label="l'indice des prix (IPC)", target="cible 2 %"),
    "en": dict(
        title="The same series, two stories",
        sub="One dropdown turns a level into an inflation rate",
        y1="« Level »", y2="« Change from year ago », %",
        cpi_label="the price index (CPI)", target="2% target"),
}

def caption(inflation: Series, lang: str) -> str:
    """Note interne dynamique : le dernier point d'inflation, dans la langue voulue."""
    latest, when = inflation.iloc[-1], inflation.index[-1]
    if lang == "fr":
        value = f"{latest:.1f}".replace(".", ",")
        return ("Le même indice des prix, vu comme « Niveau » (en haut) puis comme « Variation sur un an » (en bas) : c'est\n"
                f"ainsi qu'on lit l'inflation, à {value} % en {MONTHS_FR[when.month - 1]} {when.year} (dernier point). Source : BLS via FRED (CPIAUCSL).")
    return ("The same price index, seen as « Level » (top) then as « Change from year ago » (bottom): that is how you\n"
            f"read inflation, at {latest:.1f}% in {when:%B %Y} (latest point). Source: BLS via FRED (CPIAUCSL).")

def build_figure(cpi: Series, inflation: Series, lang: str) -> Figure:
    """Deux panneaux : le niveau de l'IPC (haut) puis sa variation annuelle (bas)."""
    text = LABELS[lang]
    fig = nm.figure(height_px=1140)
    top = fig.add_axes([0.075, 0.553, 0.907, 0.225])
    bottom = fig.add_axes([0.075, 0.140, 0.907, 0.307])

    top.plot(cpi.index, cpi, color=nm.COLORS["blue"], linewidth=3.2)
    top.set_ylabel(text["y1"])
    top.set_yticks([200, 300])
    top.text(0.065, 0.80, text["cpi_label"], transform=top.transAxes, fontsize=21.5, color=nm.COLORS["muted"])
    top.tick_params(labelbottom=False)

    bottom.plot(inflation.index, inflation, color=nm.COLORS["rose"], linewidth=3.2)
    bottom.axhline(2, color=nm.COLORS["amber"], linestyle=(0, (6, 4)), linewidth=2.6)
    bottom.axhline(0, color=nm.COLORS["muted"], linewidth=1.6, alpha=0.9)
    bottom.set_ylabel(text["y2"])
    bottom.set_yticks(range(-2, 9, 2))
    bottom.text(0.065, 0.80, text["target"], transform=bottom.transAxes, fontsize=21.5,
                fontweight="bold", color=nm.COLORS["amber"])

    for ax in (top, bottom):
        ax.set_xlim(cpi.index[0], cpi.index[-1])
        ax.margins(x=0.012)
        ax.xaxis.set_major_locator(mdates.YearLocator(5))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    nm.header(fig, text["title"], text["sub"])
    nm.footer(fig, caption(inflation, lang))
    return fig

build_figure(cpi, inflation, LANG)